In [7]:
import numpy as np
import sys
import os

# Add the project root to the path
sys.path.append('/home/justin/code/point-to-pose')

# from point2pose.modules.register.svd_register import SVDRegister
# from point2pose.modules.register.svd_residual_outlier import SVDResidualOutlierRegister
from point2pose.utils.transform import transform_pts, inverse_SE3

# ---- 1) Load the whole npz ----
D = np.load('/home/justin/code/point-to-pose/debug/pipeline/meta_data/meata_data.npz', allow_pickle=True)  # dict-like

print("Keys:", list(D.files))  # discover what's inside
N = len(D["frame_id"])        # number of rows/frames
print("Num rows:", N)

# ---- 2) Helper to unpack ragged fields ----
def unpack_ragged(name: str, store: dict, dim=-1):
    data    = store[f"{name}_data"]
    offsets = store[f"{name}_offsets"]
    lengths = store[f"{name}_lengths"]
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off:off+L]
        # Reshape to (N, 3) assuming 3D points
        if dim == 3:
            reshaped_data = flat_data.reshape(-1, 3)
        elif dim == 2:
            reshaped_data = flat_data.reshape(-1, 2)
        elif dim == -1:
            reshaped_data = flat_data
        else:
            print(f"Warning: {name} data length {len(flat_data)} not divisible by 3")
            reshaped_data = flat_data  # Keep as 1D if can't reshape
        out.append(reshaped_data)
    return out  # -> list of (N, 3) ndarrays (one per row)

# ---- 3) Access fixed-shape fields (already stacked) ----
timestamp  = D["timestamp"]          # shape (N,)
frame_id   = D["frame_id"]           # shape (N,)

print(D["reg_key_points_data"].shape)

# ---- 4) Access ragged fields ----
reg_key_points_idx_list = unpack_ragged("reg_key_points_idx", D)  # list of (Mi,) int arrays
reg_key_points_list = unpack_ragged("reg_key_points", D,dim=3)  # list of (Mi,3) float arrays
reg_cur3d_list = unpack_ragged("reg_curr3d", D,dim=3)            # list of (Mi,3) float arrays
reg_inlier_list = unpack_ragged("reg_inliers", D)          # list of (Mi,) bool arrays
track3d = unpack_ragged("track3d", D,dim=3)
visibles = unpack_ragged("visibles", D)
uncertainties = unpack_ragged("uncertainties", D)


# obj pose
obj_init_pose = D["obj_init_pose"][0]
obj_pose_list = D["obj_pose"]
obj_key_points = unpack_ragged("obj_key_points", D,dim=3)
obj_uncertainties = unpack_ragged("obj_uncertainties", D)

print(len(obj_key_points))



print(f"\nExtracted registration data:")
print(f"  reg_key_points_list: {len(reg_key_points_list)} frames")
print(f"  reg_cur3d_list: {len(reg_cur3d_list)} frames")

# Show shapes for first few frames
# for i in range(min(3, len(reg_key_points_list))):
#     print(f"  Frame {i}: reg_key_points {reg_key_points_list[i].shape}, reg_cur3d {reg_cur3d_list[i].shape}")
#     print(reg_key_points_list[i])


# Create a register instance for debugging
config = {
    'debug_level': 1,
    'debug_dir': '/home/justin/code/point-to-pose/debug/register_test'
}

print(f"\nCreated SVDRegister instance with debug level: {config['debug_level']}")

# Store the data for use in other cells
print(f"\nData loaded successfully! Available variables:")
print(f"  - D: Full data dictionary")
print(f"  - N: Number of frames ({N})")
print(f"  - frame_id, obj_id, res_mean, num_points: Fixed-shape arrays")
print(f"  - reg_key_points_list, reg_cur3d_list: Lists of point clouds")


Keys: ['timestamp', 'frame_id', 'valid', 'obj_init_pose', 'obj_pose', 'too_few_points', 'reg_iter', 'reg_thr', 'reg_key_points_idx_data', 'reg_key_points_idx_offsets', 'reg_key_points_idx_lengths', 'obj_uncertainties_data', 'obj_uncertainties_offsets', 'obj_uncertainties_lengths', 'reg_inliers_data', 'reg_inliers_offsets', 'reg_inliers_lengths', 'valid_depth_data', 'valid_depth_offsets', 'valid_depth_lengths', 'obj_valid_data', 'obj_valid_offsets', 'obj_valid_lengths', 'reg_residuals_data', 'reg_residuals_offsets', 'reg_residuals_lengths', 'reg_curr3d_data', 'reg_curr3d_offsets', 'reg_curr3d_lengths', 'visibles_data', 'visibles_offsets', 'visibles_lengths', 'obj_key_points_data', 'obj_key_points_offsets', 'obj_key_points_lengths', 'uncertainties_data', 'uncertainties_offsets', 'uncertainties_lengths', 'track2d_data', 'track2d_offsets', 'track2d_lengths', 'track3d_data', 'track3d_offsets', 'track3d_lengths', 'reg_key_points_data', 'reg_key_points_offsets', 'reg_key_points_lengths']
Num 

In [8]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
# Enable interactive mode for 3D plots
matplotlib.use('TkAgg')  # or 'Qt5Agg' depending on your system
plt.ion()  # Turn on interactive mode

def visualize_pcd(pcd):
    # Get points from your point cloud
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    # Create 3D plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot points
    if colors is not None:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c=colors, s=1, alpha=0.8)
    else:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c='blue', s=1, alpha=0.8)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Point Cloud Visualization')
    plt.show()

In [9]:
import gtsam


prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1]))
between_noise = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
)

graph = gtsam.NonlinearFactorGraph()
initial_estimate = gtsam.Values()

inserted_landmarks = set()
prev_num_kp = 0
for i in range(100):

    Xi = gtsam.symbol('x',i)
    if i == 0:
        X0 = gtsam.Pose3(inverse_SE3(obj_init_pose))
        initial_estimate.insert(Xi, X0)

        graph.push_back(gtsam.PriorFactorPose3(Xi, X0, prior_noise))

        current_estimate = initial_estimate
    else:

        Xim1 = gtsam.symbol('x',i-1)
        
        pose_i = inverse_SE3(obj_pose_list[i])
        pose_im1_inv = obj_pose_list[i - 1]
        # add initial guess for the pose
        initial_estimate.insert(Xi, gtsam.Pose3(pose_i))

        # add between factor
        between_pose = gtsam.Pose3(pose_i @ pose_im1_inv)
        graph.push_back(
            gtsam.BetweenFactorPose3(Xim1, Xi, between_pose, between_noise)
        )

    # add landmark
    cur_kp = obj_key_points[i]
    cur_kp_idx = reg_key_points_idx_list[i]
    cur_3d = reg_cur3d_list[i]
    # if len(cur_kp) > prev_num_kp:
    #     for kp in cur_kp:
    #         if kp not in inserted_landmarks:
    #             inserted_landmarks.add(kp)
    #             graph.push_back(gtsam.PriorFactorPoint3(gtsam.symbol('l', len(inserted_landmarks) - 1), kp, prior_noise))
    #     prev_num_kp = len(cur_kp)

    # # add between factor
    

    # for kp in cur_3d:
    point_noise       = gtsam.noiseModel.Isotropic.Sigma(3, 0.01)
    # Loop over measurements
    for m, lid in enumerate(cur_kp_idx):

        z_cam = cur_3d[m]                          # (3,), camera_i frame
        Lj = gtsam.symbol('l', int(lid))                 # stable landmark key by your ID

        # Seed landmark the first time we see it
        if not initial_estimate.exists(Lj):
            # world guess from current pose + camera measurement: p_w ≈ X_i * z_cam
            Xi_init = initial_estimate.atPose3(Xi)
            p_w = Xi_init.transformFrom(gtsam.Point3(*z_cam))
            initial_estimate.insert(Lj, p_w)

        # compute range and bearing
        z_range = np.linalg.norm(z_cam)
        z_bearing = gtsam.Unit3(z_cam / z_range)

        # add factor
        graph.push_back(gtsam.BearingRangeFactor3D(Xi, Lj, z_bearing, z_range, point_noise))
    
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimate, params)

# Perform the optimization
result = optimizer.optimize()
print(result)

# print("\nFactor Graph:\n{}".format(graph))


# marginals = gtsam.Marginals(graph, current_estimate)
# i = 0
# while current_estimate.exists(gtsam.symbol('x',i)):
#     print(f"X{i} covariance:\n{marginals.marginalCovariance(gtsam.symbol('x',i))}\n")
#     i += 1



Initial error: 2.2e+05, values: 203
Values with 203 values:
Value l0: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.038;
	-0.028;
	-0.0067
]

Value l1: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	-0.027;
	-0.089;
	-0.0098
]

Value l2: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.12;
	-0.064;
	0.014
]

Value l3: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.11;
	0.034;
	-0.034
]

Value l4: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	-0.039;
	0.0039;
	-0.03
]

Value l5: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.011;
	0.014;
	-0.029
]

Value l6: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	-0.013;
	-0.037;
	-0.0087
]

Value l7: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.022;
	-0.082;
	-0.0034
]

Value l8: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.091;
	-0.021;
	-0.0033
]

Value l9: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.069;
	-0.064;
	0.0047
]

Value l10: (Eigen::Matrix<double, -1, 1, 0, -1, 1>)
[
	0.059;
	0.015;
	-0.023
]

Value l11: (Eigen::Matrix<double, -1, 1, 0, -1,

In [10]:
# Extract optimized landmarks and poses
def extract_optimized_landmarks(result, max_landmark_id=1000):
    """Extract optimized landmark positions from GTSAM result"""
    landmarks = {}
    for i in range(max_landmark_id):
        Lj = gtsam.symbol('l', i)
        if result.exists(Lj):
            landmark_pos = result.atPoint3(Lj)
            landmarks[i] = np.array(landmark_pos)
    return landmarks

def extract_optimized_poses(result, num_poses=100):
    """Extract optimized pose positions from GTSAM result"""
    poses = {}
    for i in range(num_poses):
        try:
            Xi = gtsam.symbol('x', i)
            if result.exists(Xi):
                pose = result.atPose3(Xi)
                # Extract translation component
                translation = np.array([pose.x(), pose.y(), pose.z()])
                poses[i] = translation
        except:
            break
    return poses

# Extract optimized results
optimized_landmarks = extract_optimized_landmarks(result)
optimized_poses = extract_optimized_poses(result)

print(f"Found {len(optimized_landmarks)} optimized landmarks")
print(f"Found {len(optimized_poses)} optimized poses")

# Extract unoptimized landmarks (from initial estimates)
unoptimized_landmarks = extract_optimized_landmarks(initial_estimate)
unoptimized_poses = extract_optimized_poses(initial_estimate)

print(f"Found {len(unoptimized_landmarks)} unoptimized landmarks")
print(f"Found {len(unoptimized_poses)} unoptimized poses")


Found 103 optimized landmarks
Found 100 optimized poses
Found 103 unoptimized landmarks
Found 100 unoptimized poses


In [11]:
# 3D Visualization of optimized vs unoptimized results
def visualize_landmarks_and_poses(landmarks_dict, poses_dict, title, color='blue', marker='o', size=50):
    """Visualize landmarks and poses in 3D with interactive rotation"""
    fig = plt.figure(figsize=(15, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot landmarks
    if landmarks_dict:
        landmark_points = np.array(list(landmarks_dict.values()))
        ax.scatter(landmark_points[:, 0], landmark_points[:, 1], landmark_points[:, 2], 
                  c=color, marker=marker, s=size, alpha=0.8, label=f'Landmarks ({len(landmark_points)})')
    
    # Plot poses
    if poses_dict:
        pose_points = np.array(list(poses_dict.values()))
        ax.scatter(pose_points[:, 0], pose_points[:, 1], pose_points[:, 2], 
                  c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(pose_points)})')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'{title} - Interactive 3D Plot (Click and drag to rotate)')
    ax.legend()
    
    # Set equal aspect ratio for better visualization
    if landmarks_dict:
        landmark_points = np.array(list(landmarks_dict.values()))
        max_range = np.array([landmark_points[:, 0].max() - landmark_points[:, 0].min(),
                             landmark_points[:, 1].max() - landmark_points[:, 1].min(),
                             landmark_points[:, 2].max() - landmark_points[:, 2].min()]).max() / 2.0
        mid_x = (landmark_points[:, 0].max() + landmark_points[:, 0].min()) * 0.5
        mid_y = (landmark_points[:, 1].max() + landmark_points[:, 1].min()) * 0.5
        mid_z = (landmark_points[:, 2].max() + landmark_points[:, 2].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Enable interactive features
    ax.grid(True, alpha=0.3)
    
    # Add some styling for better visibility
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    
    # Make the panes transparent
    ax.xaxis.pane.set_edgecolor('w')
    ax.yaxis.pane.set_edgecolor('w')
    ax.zaxis.pane.set_edgecolor('w')
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax

# Visualize unoptimized results
print("Unoptimized Results:")
fig1, ax1 = visualize_landmarks_and_poses(unoptimized_landmarks, unoptimized_poses, 
                              "Unoptimized Landmarks and Poses", color='lightblue', marker='o')

# Visualize optimized results  
print("Optimized Results:")
fig2, ax2 = visualize_landmarks_and_poses(optimized_landmarks, optimized_poses, 
                              "Optimized Landmarks and Poses", color='darkblue', marker='o')


Unoptimized Results:
Optimized Results:


In [12]:
# Side-by-side comparison and statistics
def compare_optimization_results(unopt_landmarks, opt_landmarks, unopt_poses, opt_poses):
    """Compare unoptimized vs optimized results with interactive 3D plots"""
    
    # Create side-by-side plot
    fig = plt.figure(figsize=(20, 8))
    
    # Unoptimized plot
    ax1 = fig.add_subplot(121, projection='3d')
    if unopt_landmarks:
        unopt_landmark_points = np.array(list(unopt_landmarks.values()))
        ax1.scatter(unopt_landmark_points[:, 0], unopt_landmark_points[:, 1], unopt_landmark_points[:, 2], 
                   c='lightblue', marker='o', s=50, alpha=0.8, label=f'Landmarks ({len(unopt_landmark_points)})')
    
    if unopt_poses:
        unopt_pose_points = np.array(list(unopt_poses.values()))
        ax1.scatter(unopt_pose_points[:, 0], unopt_pose_points[:, 1], unopt_pose_points[:, 2], 
                   c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(unopt_pose_points)})')
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Unoptimized Results - Interactive 3D')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Optimized plot
    ax2 = fig.add_subplot(122, projection='3d')
    if opt_landmarks:
        opt_landmark_points = np.array(list(opt_landmarks.values()))
        ax2.scatter(opt_landmark_points[:, 0], opt_landmark_points[:, 1], opt_landmark_points[:, 2], 
                   c='darkblue', marker='o', s=50, alpha=0.8, label=f'Landmarks ({len(opt_landmark_points)})')
    
    if opt_poses:
        opt_pose_points = np.array(list(opt_poses.values()))
        ax2.scatter(opt_pose_points[:, 0], opt_pose_points[:, 1], opt_pose_points[:, 2], 
                   c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(opt_pose_points)})')
    
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Optimized Results - Interactive 3D')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Style both subplots for better visibility
    for ax in [ax1, ax2]:
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('w')
        ax.yaxis.pane.set_edgecolor('w')
        ax.zaxis.pane.set_edgecolor('w')
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax1, ax2
    
    # Calculate statistics
    print("\n" + "="*50)
    print("OPTIMIZATION STATISTICS")
    print("="*50)
    
    if unopt_landmarks and opt_landmarks:
        # Find common landmarks
        common_landmark_ids = set(unopt_landmarks.keys()) & set(opt_landmarks.keys())
        if common_landmark_ids:
            print(f"\nLandmark Analysis:")
            print(f"  Total landmarks (unoptimized): {len(unopt_landmarks)}")
            print(f"  Total landmarks (optimized): {len(opt_landmarks)}")
            print(f"  Common landmarks: {len(common_landmark_ids)}")
            
            # Calculate displacement for common landmarks
            displacements = []
            for lid in common_landmark_ids:
                unopt_pos = unopt_landmarks[lid]
                opt_pos = opt_landmarks[lid]
                displacement = np.linalg.norm(opt_pos - unopt_pos)
                displacements.append(displacement)
            
            if displacements:
                print(f"  Mean landmark displacement: {np.mean(displacements):.6f}")
                print(f"  Max landmark displacement: {np.max(displacements):.6f}")
                print(f"  Min landmark displacement: {np.min(displacements):.6f}")
    
    if unopt_poses and opt_poses:
        # Find common poses
        common_pose_ids = set(unopt_poses.keys()) & set(opt_poses.keys())
        if common_pose_ids:
            print(f"\nPose Analysis:")
            print(f"  Total poses (unoptimized): {len(unopt_poses)}")
            print(f"  Total poses (optimized): {len(opt_poses)}")
            print(f"  Common poses: {len(common_pose_ids)}")
            
            # Calculate displacement for common poses
            pose_displacements = []
            for pid in common_pose_ids:
                unopt_pos = unopt_poses[pid]
                opt_pos = opt_poses[pid]
                displacement = np.linalg.norm(opt_pos - unopt_pos)
                pose_displacements.append(displacement)
            
            if pose_displacements:
                print(f"  Mean pose displacement: {np.mean(pose_displacements):.6f}")
                print(f"  Max pose displacement: {np.max(pose_displacements):.6f}")
                print(f"  Min pose displacement: {np.min(pose_displacements):.6f}")

# Run comparison
fig_comp, ax1_comp, ax2_comp = compare_optimization_results(unoptimized_landmarks, optimized_landmarks, 
                           unoptimized_poses, optimized_poses)


In [13]:
# Alternative interactive visualization using plotly (if available)
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    
    def create_interactive_3d_plot(landmarks_dict, poses_dict, title, color='blue'):
        """Create highly interactive 3D plot using plotly"""
        
        fig = go.Figure()
        
        # Add landmarks
        if landmarks_dict:
            landmark_points = np.array(list(landmarks_dict.values()))
            fig.add_trace(go.Scatter3d(
                x=landmark_points[:, 0],
                y=landmark_points[:, 1],
                z=landmark_points[:, 2],
                mode='markers',
                marker=dict(
                    size=8,
                    color=color,
                    opacity=0.8
                ),
                name=f'Landmarks ({len(landmark_points)})',
                text=[f'Landmark {i}' for i in landmarks_dict.keys()],
                hovertemplate='<b>%{text}</b><br>' +
                            'X: %{x:.3f}<br>' +
                            'Y: %{y:.3f}<br>' +
                            'Z: %{z:.3f}<extra></extra>'
            ))
        
        # Add poses
        if poses_dict:
            pose_points = np.array(list(poses_dict.values()))
            fig.add_trace(go.Scatter3d(
                x=pose_points[:, 0],
                y=pose_points[:, 1],
                z=pose_points[:, 2],
                mode='markers',
                marker=dict(
                    size=6,
                    color='red',
                    symbol='triangle-up',
                    opacity=0.6
                ),
                name=f'Poses ({len(pose_points)})',
                text=[f'Pose {i}' for i in poses_dict.keys()],
                hovertemplate='<b>%{text}</b><br>' +
                            'X: %{x:.3f}<br>' +
                            'Y: %{y:.3f}<br>' +
                            'Z: %{z:.3f}<extra></extra>'
            ))
        
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z',
                aspectmode='data'  # Equal aspect ratio
            ),
            width=800,
            height=600
        )
        
        return fig
    
    def create_side_by_side_plotly(unopt_landmarks, opt_landmarks, unopt_poses, opt_poses):
        """Create side-by-side comparison using plotly"""
        
        # Create subplots
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
            subplot_titles=('Unoptimized Results', 'Optimized Results'),
            horizontal_spacing=0.1
        )
        
        # Unoptimized plot
        if unopt_landmarks:
            unopt_landmark_points = np.array(list(unopt_landmarks.values()))
            fig.add_trace(go.Scatter3d(
                x=unopt_landmark_points[:, 0],
                y=unopt_landmark_points[:, 1],
                z=unopt_landmark_points[:, 2],
                mode='markers',
                marker=dict(size=8, color='lightblue', opacity=0.8),
                name='Unopt Landmarks',
                showlegend=False
            ), row=1, col=1)
        
        if unopt_poses:
            unopt_pose_points = np.array(list(unopt_poses.values()))
            fig.add_trace(go.Scatter3d(
                x=unopt_pose_points[:, 0],
                y=unopt_pose_points[:, 1],
                z=unopt_pose_points[:, 2],
                mode='markers',
                marker=dict(size=6, color='red', symbol='triangle-up', opacity=0.6),
                name='Unopt Poses',
                showlegend=False
            ), row=1, col=1)
        
        # Optimized plot
        if opt_landmarks:
            opt_landmark_points = np.array(list(opt_landmarks.values()))
            fig.add_trace(go.Scatter3d(
                x=opt_landmark_points[:, 0],
                y=opt_landmark_points[:, 1],
                z=opt_landmark_points[:, 2],
                mode='markers',
                marker=dict(size=8, color='darkblue', opacity=0.8),
                name='Opt Landmarks',
                showlegend=False
            ), row=1, col=2)
        
        if opt_poses:
            opt_pose_points = np.array(list(opt_poses.values()))
            fig.add_trace(go.Scatter3d(
                x=opt_pose_points[:, 0],
                y=opt_pose_points[:, 1],
                z=opt_pose_points[:, 2],
                mode='markers',
                marker=dict(size=6, color='red', symbol='triangle-up', opacity=0.6),
                name='Opt Poses',
                showlegend=False
            ), row=1, col=2)
        
        fig.update_layout(
            title="GTSAM Optimization Results - Interactive 3D Comparison",
            width=1200,
            height=600
        )
        
        # Update both subplots to have equal aspect ratio
        for i in [1, 2]:
            fig.update_scenes(
                aspectmode='data',
                row=1, col=i
            )
        
        return fig
    
    print("Creating highly interactive 3D visualizations with plotly...")
    
    # Individual plots
    fig_unopt_plotly = create_interactive_3d_plot(unoptimized_landmarks, unoptimized_poses, 
                                                 "Unoptimized Results - Plotly 3D", 'lightblue')
    fig_opt_plotly = create_interactive_3d_plot(optimized_landmarks, optimized_poses, 
                                               "Optimized Results - Plotly 3D", 'darkblue')
    
    # Side-by-side comparison
    fig_comparison_plotly = create_side_by_side_plotly(unoptimized_landmarks, optimized_landmarks, 
                                                      unoptimized_poses, optimized_poses)
    
    # Show plots
    print("Unoptimized Results (Plotly):")
    fig_unopt_plotly.show()
    
    print("Optimized Results (Plotly):")
    fig_opt_plotly.show()
    
    print("Side-by-side Comparison (Plotly):")
    fig_comparison_plotly.show()
    
except ImportError:
    print("Plotly not available. Using matplotlib interactive plots only.")
    print("To install plotly for better 3D interactivity, run: pip install plotly")


Creating highly interactive 3D visualizations with plotly...


ValueError: 
    Invalid value of type 'builtins.str' received for the 'symbol' property of scatter3d.marker
        Received value: 'triangle-up'

    The 'symbol' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['circle', 'circle-open', 'cross', 'diamond',
            'diamond-open', 'square', 'square-open', 'x']
      - A tuple, list, or one-dimensional numpy array of the above